# Garmin — last 3 days

Pull **activities**, **sleep**, **steps**, **calories** (resting + active + total), **HRV**, and **heart rate** for the past three calendar days.

**Auth:** Uses saved tokens in `~/.garminconnect` by default. Set `GARMIN_EMAIL` / `GARMIN_PASSWORD` (or `EMAIL` / `PASSWORD`) for a fresh login. Optional: `GARMINTOKENS` to override the token path.

Run cells top to bottom.

In [ ]:
import json
import logging
import os
from datetime import date, timedelta
from getpass import getpass
from pathlib import Path
from typing import Any

from garminconnect import (
    Garmin,
    GarminConnectAuthenticationError,
    GarminConnectConnectionError,
    GarminConnectTooManyRequestsError,
)
from IPython.display import Markdown, display

logging.getLogger("garminconnect").setLevel(logging.CRITICAL)

In [ ]:
def safe_api_call(api_method, *args, **kwargs):
    """Call an API method; return (success, result, error_message)."""
    try:
        return True, api_method(*args, **kwargs), None
    except GarminConnectAuthenticationError as e:
        return False, None, f"Authentication error: {e}"
    except GarminConnectTooManyRequestsError as e:
        return False, None, f"Rate limit: {e}"
    except GarminConnectConnectionError as e:
        error_str = str(e)
        if "400" in error_str:
            return False, None, "Not available (400)"
        if "404" in error_str:
            return False, None, "Not found (404)"
        return False, None, f"Connection error: {e}"
    except Exception as e:
        return False, None, f"Unexpected error: {e}"


def init_api() -> Garmin | None:
    """Restore saved tokens or log in. Tokens: ~/.garminconnect by default."""
    tokenstore = os.getenv("GARMINTOKENS", "~/.garminconnect")
    tokenstore_path = str(Path(tokenstore).expanduser())

    try:
        garmin = Garmin()
        garmin.login(tokenstore_path)
        print(f"Logged in using saved tokens ({tokenstore_path}).")
        return garmin
    except GarminConnectTooManyRequestsError as err:
        print(f"Rate limit: {err}")
        return None
    except (GarminConnectAuthenticationError, GarminConnectConnectionError):
        print("No valid tokens — please log in.")

    while True:
        try:
            email = os.getenv("GARMIN_EMAIL") or os.getenv("EMAIL") or input("Email: ").strip()
            password = os.getenv("GARMIN_PASSWORD") or os.getenv("PASSWORD") or getpass("Password: ")
            garmin = Garmin(
                email=email,
                password=password,
                prompt_mfa=lambda: input("MFA code: ").strip(),
            )
            garmin.login(tokenstore_path)
            print(f"Login OK. Tokens saved to {tokenstore_path}")
            return garmin
        except GarminConnectAuthenticationError:
            print("Wrong credentials — try again.")
        except GarminConnectConnectionError as err:
            print(f"Connection error: {err}")
            return None
        except KeyboardInterrupt:
            return None


def dig(obj: Any, *keys, default=None):
    for key in keys:
        if not isinstance(obj, dict):
            return default
        obj = obj.get(key)
    return obj if obj is not None else default


def seconds_to_hours(seconds: int | float | None) -> float | None:
    if seconds is None:
        return None
    return round(seconds / 3600, 1)


def normalize_activity_list(raw: Any) -> list[dict]:
    if isinstance(raw, list):
        return [a for a in raw if isinstance(a, dict)]
    if isinstance(raw, dict):
        for key in ("activityList", "activities", "ActivitiesForDay"):
            value = raw.get(key)
            if isinstance(value, list):
                return [a for a in value if isinstance(a, dict)]
    return []


def map_hrv_status(raw: str | None) -> str:
    if not raw:
        return "unavailable"
    raw = raw.upper()
    if raw in {"LOW", "POOR", "UNBALANCED"}:
        return "low"
    if raw in {"HIGH"}:
        return "high"
    if raw in {"BALANCED"}:
        return "balanced"
    return raw.lower()


def activity_row(act: dict) -> dict:
    start_local = act.get("startTimeLocal") or ""
    start_time = start_local.split(" ")[1][:5] if " " in start_local else start_local[:5]
    duration = act.get("duration")
    distance = act.get("distance")
    return {
        "name": act.get("activityName"),
        "type": dig(act, "activityType", "typeKey") or "other",
        "start": start_time,
        "duration_min": round(duration / 60) if duration else 0,
        "calories": act.get("calories") or 0,
        "avg_hr": act.get("avgHR") or act.get("averageHR") or 0,
        "distance_km": round(distance / 1000, 2) if distance else None,
    }

In [ ]:
api = init_api()
if not api:
    raise SystemExit("Could not authenticate")

In [ ]:
END_DATE = date.today()
START_DATE = END_DATE - timedelta(days=2)
DATE_RANGE = [
    (START_DATE + timedelta(days=i)).isoformat()
    for i in range((END_DATE - START_DATE).days + 1)
]

print(f"Date range: {DATE_RANGE[0]} → {DATE_RANGE[-1]} ({len(DATE_RANGE)} days)")

In [ ]:
def fetch_day(api: Garmin, date_str: str) -> dict:
    """Fetch steps, calories, sleep, HRV, heart rate, and activities for one day."""
    _, summary, summary_err = safe_api_call(api.get_user_summary, date_str)
    _, sleep, sleep_err = safe_api_call(api.get_sleep_data, date_str)
    _, hr, hr_err = safe_api_call(api.get_heart_rates, date_str)
    _, hrv, hrv_err = safe_api_call(api.get_hrv_data, date_str)
    _, activities_raw, activities_err = safe_api_call(
        api.get_activities_by_date, date_str, date_str
    )

    summary = summary or {}
    dto = dig(sleep, "dailySleepDTO") or {}
    hrv_summary = dig(hrv, "hrvSummary") or {}
    activities = [activity_row(a) for a in normalize_activity_list(activities_raw)]

    resting_hr = (hr or {}).get("restingHeartRate") or summary.get("restingHeartRate")

    return {
        "date": date_str,
        "errors": {
            "summary": summary_err,
            "sleep": sleep_err,
            "heart_rate": hr_err,
            "hrv": hrv_err,
            "activities": activities_err,
        },
        "steps": summary.get("totalSteps"),
        "step_goal": summary.get("dailyStepGoal"),
        "active_calories": summary.get("activeKilocalories"),
        "resting_calories": summary.get("bmrKilocalories"),
        "total_calories": summary.get("totalKilocalories"),
        "sleep_hours": seconds_to_hours(dto.get("sleepTimeSeconds")),
        "deep_sleep_hours": seconds_to_hours(dto.get("deepSleepSeconds")),
        "light_sleep_hours": seconds_to_hours(dto.get("lightSleepSeconds")),
        "rem_sleep_hours": seconds_to_hours(dto.get("remSleepSeconds")),
        "awake_hours": seconds_to_hours(dto.get("awakeSleepSeconds")),
        "sleep_score": dig(dto, "sleepScores", "overall", "value"),
        "sleep_start": dto.get("sleepStartTimestampLocal"),
        "sleep_end": dto.get("sleepEndTimestampLocal"),
        "resting_hr": resting_hr,
        "min_hr": (hr or {}).get("minHeartRate") or summary.get("minHeartRate"),
        "max_hr": (hr or {}).get("maxHeartRate") or summary.get("maxHeartRate"),
        "avg_hr_7d": (hr or {}).get("lastSevenDaysAvgRestingHeartRate"),
        "sleep_avg_hr": dto.get("avgHeartRate"),
        "hrv": hrv_summary.get("lastNightAvg"),
        "hrv_status": map_hrv_status(hrv_summary.get("status")),
        "hrv_weekly_avg": hrv_summary.get("weeklyAvg"),
        "hrv_baseline_low": hrv_summary.get("baselineBalancedLow"),
        "hrv_baseline_high": hrv_summary.get("baselineBalancedHigh"),
        "activities": activities,
    }


days = [fetch_day(api, d) for d in DATE_RANGE]

In [ ]:
def print_table(title: str, headers: list[str], rows: list[list[Any]]) -> None:
    display(Markdown(f"### {title}"))
    if not rows:
        print("(no data)")
        return
    widths = [len(h) for h in headers]
    for row in rows:
        for i, cell in enumerate(row):
            widths[i] = max(widths[i], len(str(cell)))
    fmt = "  ".join(f"{{:<{w}}}" for w in widths)
    print(fmt.format(*headers))
    print(fmt.format(*["-" * w for w in widths]))
    for row in rows:
        print(fmt.format(*[str(c) if c is not None else "—" for c in row]))
    print()

In [ ]:
# Steps
step_rows = [
    [d["date"], d["steps"], d["step_goal"]]
    for d in days
]
print_table("Steps", ["date", "steps", "goal"], step_rows)

In [ ]:
# Calories — resting (BMR) + active + total
calorie_rows = [
    [
        d["date"],
        d["resting_calories"],
        d["active_calories"],
        d["total_calories"],
    ]
    for d in days
]
print_table(
    "Calories (kcal)",
    ["date", "resting (BMR)", "active", "total"],
    calorie_rows,
)

In [ ]:
# Heart rate
hr_rows = [
    [
        d["date"],
        d["resting_hr"],
        d["min_hr"],
        d["max_hr"],
        d["avg_hr_7d"],
        d["sleep_avg_hr"],
    ]
    for d in days
]
print_table(
    "Heart rate (bpm)",
    ["date", "resting", "min", "max", "7-day avg resting", "sleep avg"],
    hr_rows,
)

In [ ]:
# HRV
hrv_rows = [
    [
        d["date"],
        d["hrv"],
        d["hrv_status"],
        d["hrv_weekly_avg"],
        d["hrv_baseline_low"],
        d["hrv_baseline_high"],
    ]
    for d in days
]
print_table(
    "HRV (ms)",
    ["date", "last night", "status", "weekly avg", "baseline low", "baseline high"],
    hrv_rows,
)

In [ ]:
# Sleep
sleep_rows = [
    [
        d["date"],
        d["sleep_hours"],
        d["deep_sleep_hours"],
        d["rem_sleep_hours"],
        d["light_sleep_hours"],
        d["sleep_score"],
    ]
    for d in days
]
print_table(
    "Sleep",
    ["date", "total (h)", "deep (h)", "REM (h)", "light (h)", "score"],
    sleep_rows,
)

for d in days:
    if d["sleep_start"] or d["sleep_end"]:
        print(f"  {d['date']}: {d['sleep_start'] or '—'} → {d['sleep_end'] or '—'}")

In [ ]:
# Activities
display(Markdown("### Activities"))
for d in days:
    print(f"\n{d['date']} — {len(d['activities'])} workout(s)")
    if not d["activities"]:
        print("  (none)")
        continue
    for act in d["activities"]:
        dist = f", {act['distance_km']} km" if act["distance_km"] else ""
        print(
            f"  • {act['start']} {act['name']} ({act['type']}) — "
            f"{act['duration_min']} min, {act['calories']} kcal, "
            f"avg HR {act['avg_hr']}{dist}"
        )

In [ ]:
# Raw JSON (optional — uncomment to inspect full payloads)
# print(json.dumps(days, indent=2, default=str))